# 09 — Routing & Query Construction (Logical, Semantic & Metadata Filters)

**By the end of this notebook, you can route queries with Pydantic structured schemas or embedding cosine similarity, construct metadata filters, and log wrong-route edge cases.**

### Learning Objectives
- Implement Logical Routing with Pydantic v2 (`.with_structured_output()`) and Semantic Routing with pure cosine similarity.
- Translate natural language constraints into executable vector metadata filters cleanly (guarding malformed inputs).
- Demonstrate a wrong-route failure, log routing decisions, and know when NOT to route.

### Prerequisites
- Notebooks 00–08

## 1. Setup
Load `.env` configuration via `course/utils/models.py` and initialize our shared LLM and embeddings helpers.

In [1]:
# If running in Google Colab, uncomment: !git clone https://github.com/rahat4528/rag-from-scratch-course-v2.git && %cd rag-from-scratch-course-v2 && pip install -q -r course/requirements.txt
import sys
from pathlib import Path
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'course' / 'utils').exists()), Path.cwd())
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
from course.utils.models import get_llm, get_embeddings, resolve_course_root
course_dir = resolve_course_root(); data_dir = course_dir / 'data'
llm = get_llm(fallback_to_demo=True); embeddings = get_embeddings()
print(f'Ready | LLM={type(llm).__name__} | Embeddings={type(embeddings.base).__name__}')

[OfflineDemoChatModel] Notice: Provider 'ollama' not active (Ollama server is not reachable at http://localhost:11434. Install/start Ollama (`ollama pull llama3.2`) or switch RAG_PROVIDER in .env.). Running code-path demo mode.


W1112 01:29:02.681000 3120 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready | LLM=OfflineDemoChatModel | Embeddings=LocalSentenceTransformerEmbeddings


## 2. Concept: Hospital Triage Receptionist: Routing Before Retrieval

Imagine asking an orthopedic surgeon about payroll forms! Searching all databases equally blurs terminology and slows search.

1. **Logical Routing**: Uses structured Pydantic fields (`datasource: Literal['handbook', 'faq', 'changelog', 'safety']`) so the LLM classifies intent cleanly.
2. **Semantic Routing**: Zero LLM token cost! Embeds each domain description once and matches query cosine similarity in `<2ms`.
3. **Query Construction**: Converts natural constraints (*"from changelog"* / *"executive docs"*) into structured vector `filter` predicates.

**When NOT to use Routing:** Cross-document synthesis questions (*"Which tier includes ZeroGravity AND what does it cost?"*) span multiple files — hard-routing to ONE file deletes half the answer!

```
User Q ---> [Router (Logical or Semantic) + Decision Logger]
                 |-- 'company_handbook' --> filtered search (HR/Pricing)
                 |-- 'product_faq'      --> filtered search (ROS2/Errors)
                 |-- 'recent_changelog' --> filtered search (Oct 2026)
                 +-- 'WRONG ROUTE!' ------> Zero matching hits -> Fallback/All-corpus safety net!
```

## 3. Step-by-Step Build

### Step 1: Index Multi-Source Corpus & Logical Pydantic Router
Let's index all corpus sources with `doc_type` metadata and build a Pydantic v2 structured `RouteDecision` router.

In [2]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from course.utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=450, chunk_overlap=60).split_documents(load_corpus_documents(data_dir))
store = InMemoryVectorStore.from_documents(chunks, embeddings)

class RouteDecision(BaseModel):
    datasource: Literal["company_handbook", "product_faq", "recent_changelog", "safety_brief"] = Field(description="Target doc bucket")

logical_router = llm.with_structured_output(RouteDecision)
print("Structured route for error code:", logical_router.invoke("How do I fix ERR_QUANTUM_TIMEOUT?").datasource)
        

Structured route for error code: product_faq


> **What just happened?**
> Native Pydantic v2 validation guarantees only known datasources can ever be returned.

### Step 2: Zero-LLM Semantic Routing + Structured Decision Logger
Embed three crisp domain descriptions once, match queries by dot product, and append every routing decision to an audit log.

In [3]:
import numpy as np
route_prompts = {
    "company_handbook": "Travel meal reimbursement, Zurich hub pet policy, learning stipend, commercial subscription pricing.",
    "product_faq": "Technical troubleshooting ERR_QUANTUM_TIMEOUT port 9872, ROS2 compatibility, ZeroGravity SLA uptime.",
    "recent_changelog": "October 2026 release v4.6, HyperSync latency 42%, Chief AI Ethics Officer, REST API v1 deprecation.",
}
route_vecs = {k: np.array(embeddings.embed_query(v)) for k, v in route_prompts.items()}
routing_log = []

def semantic_route(q: str):
    qv = np.array(embeddings.embed_query(q))
    scores = {k: float(np.dot(qv, rv)) for k, rv in route_vecs.items()}
    winner = max(scores, key=scores.get)
    routing_log.append({"query": q, "route": winner, "score": round(scores[winner], 3)})
    return winner, scores[winner]
        

> **What just happened?**
> Semantic routing logs exact confidence scores alongside the winning datasource with zero LLM calls.

### Step 3: Query Construction (Natural Constraint -> Metadata Filter) + Guarding Malformed Output
Let's build a helper translating datasource names to strict store metadata filters (`doc_type`), with safe fallback if an unknown key arrives.

In [4]:
SOURCE_MAP = {"company_handbook": "handbook", "product_faq": "faq", "recent_changelog": "changelog", "safety_brief": "safety"}

def construct_filtered_search(query: str, predicted_route: str, k: int = 2):
    dtype = SOURCE_MAP.get(predicted_route)
    if not dtype:
        return store.similarity_search(query, k=k)  # Safe fallback on malformed/unknown route!
    return store.similarity_search(query, k=k, filter=lambda d: d.metadata.get("doc_type") == dtype)

route, sc = semantic_route("What is REST API v1 shutdown date?")
print("Filtered search hit:", construct_filtered_search("What is REST API v1 shutdown date?", route, k=1)[0].metadata)
        

Filtered search hit: {'source': 'recent_changelog.md', 'doc_type': 'changelog', 'access_level': 'public', 'doc_id': 'doc-changelog'}


> **What just happened?**
> Mapping unknown/malformed router outputs cleanly to unfiltered search prevents application crashes!

### Step 4: Demonstrate a Wrong-Route Failure vs. Baseline (and Why Routing Logs Matter)
For cross-document Q05 (`ZeroGravity tier AND 40ms threshold`), routing strictly to `product_faq` misses Starter/Enterprise pricing nuances in `company_handbook`!

In [5]:
q_cross = "Which tier is required for ZeroGravity Failover and what is the 40ms threshold?"
routed_only_faq = construct_filtered_search(q_cross, "product_faq", k=2)
unfiltered = store.similarity_search(q_cross, k=4)
print("Wrong hard-route sources:", sorted({d.metadata["source"] for d in routed_only_faq}))
print("Multi-doc unfiltered    :", sorted({d.metadata["source"] for d in unfiltered}))
print("Audit log snapshot      :", routing_log)
        

Wrong hard-route sources: ['product_faq.md']
Multi-doc unfiltered    : ['company_handbook.txt', 'product_faq.md']
Audit log snapshot      : [{'query': 'What is REST API v1 shutdown date?', 'route': 'recent_changelog', 'score': 0.486}]


> **What just happened?**
> Hard single-bucket routing broke cross-document retrieval — logging routes lets engineers spot when queries need multi-route or unfiltered search!

## 4. Try It Yourself
1. **Test malformed route safety** — *Hint:* Call `construct_filtered_search('Enterprise Plan cost', 'non_existent_bucket', k=2)` and verify clean fallback.
2. **Inspect semantic score on pet policy** — *Hint:* Call `semantic_route('Can dogs visit the Zurich office?')` and check winning bucket.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>
Run the cell below to check one working solution.
</details>

In [6]:
assert len(construct_filtered_search("Enterprise Plan cost", "invalid_route_xyz", k=2)) == 2
print("Pets semantic route:", semantic_route("Can dogs visit the Zurich office?"))
        

Pets semantic route: ('company_handbook', 0.49569790229528393)


## 5. Common Mistakes
- Applying single-source hard routing to multi-document comparison questions.
- Crashing when structured/parsed metadata filter output emits unexpected values — always fall back cleanly.
- Skipping decision logging (`query`, `route`, `confidence`), leaving silent misroutes invisible.

## 6. Recap
- Logical routing enforces Pydantic v2 output schemas; semantic routing uses fast vector dot products.
- Query construction converts natural intent into safe vector-store `filter` predicates.
- Wrong-route failures highlight when NOT to route: cross-document synthesis & ambiguous queries.

## 7. Next Notebook
Next (`10_advanced_indexing.ipynb`, Phase 3!): Upgrade indexing itself using Parent-Document hydration, Multi-Representation summaries, Hybrid BM25, RAPTOR hierarchical summaries, and ColBERT late interaction!

## 8. Video Chapter Outline
1. **Routing Architecture Overview** — Logical vs Semantic vs Query Construction.
2. **Pydantic v2 Structured Router** — Strict datasources without pydantic_v1 deprecation.
3. **Semantic Router + Audit Logging** — Zero-LLM triage with score tracking.
4. **Metadata Filter Builder & Malformed Guard** — Safe predicate execution.
5. **Wrong-Route Failure Demo** — When single-bucket routing hurts multi-doc answers.